[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/05-full-projects/ml-fullproj-bond.ipynb)

# Full Project: Municipal Bond Investment Risk

*AIBits Academy · Machine Learning End To End · Full Project · US Data*

Kern County, California municipal bond records — where one-hot encoding a handful of high-cardinality columns explodes 38 features into 1,302, and Linear Regression's R² collapses to −2.5 million.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Setup

In [ ]:
# Fetch the lesson's dataset(s) into the working folder
import os, io, zipfile, urllib.request, urllib.parse

DATA_BASE = "https://raw.githubusercontent.com/aimldstejas/aibits-genai-notebooks/main/ml/data/"   # course copies live in the notebooks repo
for f in ['Kern_County_bond_dataset.csv']:
    if not os.path.exists(f):
        urllib.request.urlretrieve(DATA_BASE + urllib.parse.quote(f), f)
        print('downloaded', f)

In [ ]:
# Imports used throughout this project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, mean_squared_error, mean_absolute_error, r2_score)

> **Business Problem**
>
> Municipal bond analysts want an early, automated read on a newly issued bond's likely credit quality — approximated here by predicting its **S&P Rating** — using administrative and structural fields (issuer type, debt type, sale type, principal amount, issuance costs) that are available immediately at issuance, well before a formal rating agency opinion is finalized. This dataset is US municipal bond data from Kern County, California and is presented in its native USD/US administrative context, not adapted to Indian markets.

> **Dataset**
>
> **1,200 Kern County municipal bond records, 55 raw columns.** Four columns (`Lender`, `Co-Financial Advisor`, `Co-Bond Counsel`, `Borrower Counsel`) are entirely or almost entirely empty and dropped, leaving 38 usable columns of financial figures (Principal Amount, New Money, Refunding Amount, Interest Rates) and administrative categoricals (Issuer Type, Underwriter, Bond Counsel, Sale Type). Target: `S and P Rating` (51 distinct rating grades).

## Step 1 — Impute Missing Values Column-by-Column

Most columns have some missing data — ratings columns for bonds not yet reported, advisor/counsel fields that simply don't apply to every issuance. Each column is imputed individually: mode for categoricals, mean for numerics.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("Kern_County_bond_dataset.csv")
print(df.shape)  # (1200, 55)

# Four columns are 100% / 99.8% missing — drop rather than impute
df = df.drop(columns=["CDIAC Number","Issuer","MKR CDIAC Number","Fitch Rating",
                       "Lender","Co-Financial Advisor","Co-Bond Counsel","Borrower Counsel"])

# Mode-fill categoricals, mean-fill numerics, column by column
for col in ["Debt Policy","Project Name","Purpose","Interest Type","S and P Rating","Guarantor"]:
    df[col] = df[col].fillna(df[col].mode().values[0])
for col in ["Refunding Amount","TIC Interest Rate","NIC Interest Rate","Total Issuance Costs"]:
    df[col] = df[col].fillna(df[col].mean())
for col in df.columns[df.isnull().any()]:   # any remaining gaps: same rule
    df[col] = df[col].fillna(df[col].mode().values[0] if df[col].dtype == object or str(df[col].dtype) == 'str' else df[col].mean())
print(df.isnull().sum().sum(), "missing values remaining")

## Step 2 — Encode Categoricals: the Dimensionality Trap

Low-cardinality categoricals are label-encoded. High-cardinality ones — `Underwriter` (138 unique values), `Bond Counsel` (65), `Disclosure Counsel` (22), plus every distinct `First Optional Call Date` and `Final Maturity Date` — are one-hot encoded with `pd.get_dummies()`.

Two rows are exact duplicates, and the low-cardinality text columns are **label-encoded** (as the text says). The five high-cardinality columns are left as text for the one-hot step below; the target `S and P Rating` is label-encoded too.

In [ ]:
from sklearn.preprocessing import LabelEncoder

df = df.drop_duplicates().reset_index(drop=True)
ohe_cols = ["Underwriter", "Disclosure Counsel", "Trustee", "First Optional Call Date", "Final Maturity Date"]
for col in df.select_dtypes(exclude='number').columns:
    if col not in ohe_cols:
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))
print(df.shape)

In [ ]:
ohe = pd.get_dummies(df, columns=["Underwriter","Disclosure Counsel","Trustee",
                                   "First Optional Call Date","Final Maturity Date"])
df = ohe
print(df.shape)

> **1,198 Rows, 1,302 Columns**
>
> Blanket one-hot encoding of every high-cardinality column just turned a comfortably wide dataset (38 columns, 1,198 rows — a healthy 31:1 row-to-column ratio) into one with **more columns than rows**. Each unique `First Optional Call Date` or `Underwriter` becomes its own all-or-nothing binary column, most of them populated by a single bond. This is the same curse-of-dimensionality territory covered on the PCA page, arrived at here by an encoding choice rather than genuinely high-dimensional raw data.

## Step 3 — What Happens to Plain Linear Regression

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

x = df.drop("S and P Rating", axis=1)
y = df["S and P Rating"]
x_train, x_test, y_train, y_test = train_test_split(x, y, train_size=0.7, random_state=10)

lr = LinearRegression().fit(x_train, y_train)
r2 = r2_score(y_test, lr.predict(x_test))
print("Linear Regression R²:", r2)

> **An R² of Negative 2.5 Million**
>
> This is not a typo and not a bug — it's what ordinary least squares does when the feature matrix is severely **rank-deficient**: with 1,302 columns competing to explain 1,198 rows, many one-hot columns are near-duplicates or perfectly separate a handful of bonds, making XᵀX close to singular. The closed-form solution (XᵀX)⁻¹Xᵀy still technically computes, but the resulting coefficients are enormous and essentially meaningless — predictions on unseen test bonds swing wildly out of any sane range. R² compares model error to a trivial "always predict the mean" baseline; a model this unstable can be *astronomically worse* than the trivial baseline, which is exactly what a large negative R² signals.

## Step 4 — Regularization Rescues It

In [ ]:
from sklearn.linear_model import Ridge, Lasso

ridge = Ridge().fit(x_train, y_train)
print("Ridge R²:  ", r2_score(y_test, ridge.predict(x_test)))

lasso = Lasso().fit(x_train, y_train)
print("Lasso R²:  ", r2_score(y_test, lasso.predict(x_test)))

Adding a single penalty term — nothing else changes about the data or features — takes R² from **−2,545,768** to **0.598**. This is the most dramatic real-data illustration in this course of why Ridge and Lasso regression exist: once p approaches or exceeds n, unregularized linear models don't just underperform, they can fail catastrophically.

## Step 5 — Tree Ensembles, and a GridSearchCV That Backfires

**Heads-up:** the grid search below fits 8 random forests x 5 folds; expect a few minutes on Colab's free CPU.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

rfr = RandomForestRegressor().fit(x_train, y_train)
print("Random Forest R² (default):", r2_score(y_test, rfr.predict(x_test)))

params = {"max_depth":[100,200], "min_samples_split":[3,5],
          "n_estimators":[200,500], "bootstrap":[True], "oob_score":[True]}
rfrt = GridSearchCV(RandomForestRegressor(), params, cv=5, scoring="r2")
rfrt.fit(x_train, y_train)
print("Random Forest R² (tuned, CV):", rfrt.best_score_)

> **Tuning Made It Worse**
>
> The default, untuned Random Forest (R²=0.805) beat every configuration the grid search tried (best CV R²=0.726). This is not a contradiction — `best_score_` is a 5-fold cross-validated average *within the training set*, computed on a dataset with only 1,198 rows split five ways (roughly 240 rows per fold). With that little data per fold, cross-validated scores carry real variance, and a search space that happened to exclude the specific default configuration (or nearby settings) can easily report a lower average than the lucky single train/test split used for the "default" number above. The lesson mirrors the Credit Card Lead Prediction full project: **GridSearchCV only ever reports the best result within the space and folds you gave it** — it is not a guarantee of improvement over sensible defaults, especially on small datasets.

| Model | R² Score |
|---|---|
| Linear Regression (default) | −2,545,768.65 |
| Ridge Regression (default) | 0.598 |
| Lasso Regression (default) | 0.480 |
| AdaBoost Regressor (default) | 0.581 → 0.652 tuned |
| Decision Tree Regressor (default) | 0.649 |
| Random Forest Regressor (default) | **0.805** |
| Random Forest Regressor (GridSearchCV tuned) | 0.726 (worse) |

## Visualizing the Dimensionality Trap

Plain Linear Regression's R² is so catastrophically negative it can't share an axis with anything else — the broken bar below shows it truncated, with the real number labelled. Everything else fits on a normal 0–1 scale, including the two models where tuning went in opposite directions.

## Key Business Takeaways

- Blanket one-hot encoding of every categorical column, regardless of cardinality, is rarely the right default — a 65-category or 138-category column deserves target encoding, grouping of rare categories, or dimensionality reduction, not automatic expansion into hundreds of binary columns.
- Once feature count approaches or exceeds row count, unregularized linear models aren't just weaker — they can be numerically catastrophic. Ridge/Lasso should be the default starting point in high-dimensional settings, not an afterthought.
- Hyperparameter tuning is not guaranteed to help, especially on small datasets where cross-validation folds are themselves noisy estimates — always compare the tuned result against a sensible default before trusting `best_score_` blindly.

## Practice Questions

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · More columns than rows

Store in `p_over_n` the ratio of feature columns to training rows (`x_train.shape[1] / x_train.shape[0]`) and `wide` = whether it exceeds 1. This is why plain OLS collapses.

In [ ]:
p_over_n = wide = None   # TODO


In [ ]:
try:
    check("more features than rows", wide is True)
    check("ratio", abs(p_over_n - x_train.shape[1] / x_train.shape[0]) < 1e-12)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
p_over_n = x_train.shape[1] / x_train.shape[0]
wide = bool(p_over_n > 1)

```

</details>

### Exercise 2 · Medium · Lasso zeroes coefficients

Using the lesson's fitted `lasso`, store the number of coefficients that are exactly zero in `n_zero` and the number that survive in `n_kept`.

In [ ]:
n_zero = n_kept = None   # TODO


In [ ]:
try:
    check("adds up", n_zero + n_kept == x_train.shape[1])
    check("most features are dropped", n_zero > n_kept)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
n_zero = int((lasso.coef_ == 0).sum())
n_kept = int((lasso.coef_ != 0).sum())

```

</details>

### Exercise 3 · Stretch · Choose Ridge's alpha by cross-validation

Fit `RidgeCV(alphas=[0.1, 1, 10, 100, 1000])` on the training data. Store the selected alpha in `best_alpha` and the test R² in `ridge_cv_r2`.

In [ ]:
from sklearn.linear_model import RidgeCV
best_alpha = ridge_cv_r2 = None   # TODO


In [ ]:
try:
    check("alpha from the grid", best_alpha in [0.1, 1, 10, 100, 1000])
    check("regularised model works far better than plain OLS", ridge_cv_r2 > 0.3)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.linear_model import RidgeCV
rcv = RidgeCV(alphas=[0.1, 1, 10, 100, 1000]).fit(x_train, y_train)
best_alpha = rcv.alpha_
ridge_cv_r2 = r2_score(y_test, rcv.predict(x_test))

```

</details>

---
*Back to the course: **Machine Learning End To End → Full Project: Municipal Bond Investment Risk**.*